# 🧩 Полный анализ данных: отчёты, граф зависимостей

Этот ноутбук выполняет:
1. Загрузку CSV‑файлов (кодировка cp1251, разделитель `;`).
2. Очистку и обогащение данных (создание `Full Name`, `Ingoing References` и т.д.).
3. Генерацию текстовых отчётов (через кнопки скачивания).

## Загрузка данных

1. Перетащите ваши CSV-файлы (`Cubes.csv`, `Multicubes.csv`, `Lists.csv`, `ListsProperties.csv`, `CubesSubsets.csv`, `ListsSubsets.csv`) в боковую панель JupyterLite (левая часть экрана, область "Files").
2. После того как файлы появятся в списке, выполните следующую ячейку — она прочитает их с правильной кодировкой cp1251.

#### Критическое значение количества клеток для куба

В следующей ячейке можно задать значение минимального количества клеток для кубов, которые попадут в специализированные отчеты.

Например, если вы хотите получить отчет о кубах, где количество ячеек больше 100 тысяч, задайте следующее значение:

```py
cube_threshold_value = 100_000
```

Значение по умолчанию - 500 тысяч ячеек.

In [ ]:
cube_threshold_value = 500_000

In [ ]:
# === 0. Установка дополнительных пакетов (выполняется один раз в сессии) ===
import piplite
await piplite.install('networkx')
await piplite.install('matplotlib')
await piplite.install('ipywidgets')
print("✅ Пакеты networkx, matplotlib, ipywidgets установлены")

In [ ]:
# === 1. Импорт библиотек
import pandas as pd
import numpy as np
import re
import io
import sys
import base64
from collections import defaultdict
import networkx as nx
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML

In [ ]:
# === 2. Функция для скачивания отчета
def download_text(content, filename):
    b64 = base64.b64encode(content.encode('utf-8')).decode()
    href = f'<a download="{filename}" href="data:text/plain;charset=utf-8;base64,{b64}">📥 Скачать {filename}</a>'
    display(HTML(href))

In [ ]:
# === 3. Чтение всех исходных данных
def read_cp1251_csv(filename, sep=';'):
    with open(filename, 'rb') as f:
        raw = f.read()
    text = raw.decode('cp1251')
    # Удаляем BOM, если он есть (не повредит)
    if text.startswith('\ufeff'):
        text = text[1:]
    return pd.read_csv(io.StringIO(text), sep=sep)

# Список ожидаемых файлов (можно оставить только те, что нужны)
needed_files = ['Cubes.csv', 'Multicubes.csv', 'Lists.csv', 'ListsProperties.csv', 'CubesSubsets.csv', 'ListsSubsets.csv']
dataframes = {}

for fname in needed_files:
    try:
        df = read_cp1251_csv(fname)
        dataframes[fname] = df
        print(f"✅ Загружен {fname} — строк: {len(df)}")
    except FileNotFoundError as e:
        print(f"⚠️ {e}")
    except Exception as e:
        print(f"❌ Ошибка при чтении {fname}: {e}")

# Проверка, что все нужные файлы загружены
if len(dataframes) < len(needed_files):
    print("\n❗ Некоторые файлы не загружены. Пожалуйста, перетащите недостающие CSV.")
else:
    print("\n✅ Все файлы успешно загружены. Можно выполнять следующие ячейки.")

## 📊 Часть 1: Очистка и обогащение данных

In [ ]:
def rename_first_column(df, new_name):
    """Переименовывает первый столбец DataFrame в new_name"""
    if len(df.columns) > 0:
        cols = list(df.columns)
        cols[0] = new_name
        df.columns = cols
    return df

# === Функции очистки (оригинал из model_data) ===
def clean_df_cubes(df):
    """Очистка Cubes: удаление пустых Multicube, создание Full Name"""
    df = df.dropna(subset=['Multicube'])
    df = df[df['Multicube'].str.strip() != '']
    df['Cubes'] = df['Cubes'].fillna('').str.strip()
    df['Multicube'] = df['Multicube'].str.strip()
    df['Full Name'] = df['Multicube'] + '.' + df['Cubes']
    df = df[df['Full Name'] != '.']
    
    def quote_if_needed(val):
        if pd.isna(val) or val == '':
            return "''"
        if isinstance(val, str) and re.fullmatch(r'[a-zA-Z0-9_]+', val):
            return val
        else:
            return f"'{val}'"
    
    processed_multicube = df['Multicube'].apply(quote_if_needed)
    processed_cubes = df['Cubes'].apply(quote_if_needed)
    df['Full Name'] = processed_multicube + '.' + processed_cubes
    cols = list(df.columns)
    multicube_idx = cols.index('Multicube')
    cols.insert(multicube_idx + 1, cols.pop(cols.index('Full Name')))
    df = df[cols]
    return df

def clean_df_multicubes(df):
    """Очистка Multicubes: удаление строк с пустыми User Lists, Time Scale и Cell Count == 0"""
    required = ['User Lists', 'Time Scale', 'Cell Count', 'Multicubes']
    for col in required:
        if col not in df.columns:
            df[col] = ''
    df['Cell Count'] = pd.to_numeric(df['Cell Count'], errors='coerce').fillna(0)
    mask = (df['User Lists'].isna() | (df['User Lists'] == '')) & \
           (df['Time Scale'].isna() | (df['Time Scale'] == '')) & \
           (df['Cell Count'] == 0)
    df = df[~mask].reset_index(drop=True)
    
    def process_name(val):
        if pd.isna(val) or val == '':
            return ''
        if re.fullmatch(r'[\w]+', str(val)):
            return str(val)
        else:
            return f"'{val}'"
    df['Full Name'] = df['Multicubes'].apply(process_name)
    cols = list(df.columns)
    if 'Full Name' in cols:
        cols.remove('Full Name')
        cols.insert(1, 'Full Name')
        df = df[cols]
    return df

def clean_df_lists(df):
    """Очистка Lists: удаление строк с Cell Count == 0 и Element Count == 0"""
    df['Cell Count'] = pd.to_numeric(df['Cell Count'], errors='coerce').fillna(0)
    df['Element Count'] = pd.to_numeric(df['Element Count'], errors='coerce').fillna(0)
    mask = (df['Cell Count'] == 0) & (df['Element Count'] == 0)
    df = df[~mask].reset_index(drop=True)
    def process_name(val):
        if pd.isna(val) or val == '':
            return ''
        if re.fullmatch(r'[\w]+', str(val)):
            return str(val)
        else:
            return f"'{val}'"
    df['Full Name'] = df['Lists'].apply(process_name)
    cols = list(df.columns)
    if 'Full Name' in cols:
        cols.remove('Full Name')
        cols.insert(1, 'Full Name')
        df = df[cols]
    return df

def clean_df_listsproperties(df):
    """Очистка ListsProperties: создание List и Full Name на основе Format и ListsProperties"""
    required = ['Format', 'ListsProperties']
    for col in required:
        if col not in df.columns:
            df[col] = ''
    df = df.copy()
    def quote_if_needed(val):
        if pd.isna(val) or val == '':
            return "''"
        if isinstance(val, str) and re.fullmatch(r'[a-zA-Z0-9_]+', val):
            return val
        else:
            return f"'{val}'"
    df['List'] = ''
    df['Full Name'] = ''
    non_empty = df[df['Format'].notna() & (df['Format'] != '')].index
    for idx in non_empty:
        prev = idx - 1
        while prev >= 0 and (pd.isna(df.loc[prev, 'Format']) or df.loc[prev, 'Format'] == ''):
            prev -= 1
        if prev >= 0:
            val_prev = df.loc[prev, 'ListsProperties']
            val_curr = df.loc[idx, 'ListsProperties']
            df.loc[idx, 'List'] = val_prev
            df.loc[idx, 'Full Name'] = f"{quote_if_needed(val_prev)}.{quote_if_needed(val_curr)}"
    cols = list(df.columns)
    first = cols[0]
    for col in ['List', 'Full Name']:
        if col in cols:
            cols.remove(col)
    cols.insert(cols.index(first)+1, 'List')
    cols.insert(cols.index(first)+2, 'Full Name')
    df = df[cols]
    df = df[df['Format'].notna() & (df['Format'] != '')].reset_index(drop=True)
    return df

def clean_df_сubesubsets(df):
    """Очистка CubesSubsets: создание Full Name"""
    def process(val):
        if pd.isna(val) or val == '':
            return ''
        if re.fullmatch(r'[\w]+', str(val)):
            return str(val)
        else:
            return f"'{val}'"
    df['Full Name'] = df['CubesSubsets'].apply(process)
    cols = list(df.columns)
    if 'Full Name' in cols:
        cols.remove('Full Name')
        cols.insert(1, 'Full Name')
        df = df[cols]
    return df

def clean_referenced_by(df, context_col='List'):
    """Обработка столбца Referenced By → Outgoing References"""
    if 'Referenced By' not in df.columns:
        df['_Out Refs'] = [[] for _ in range(len(df))]
        df['Outgoing References'] = [[] for _ in range(len(df))]
        return df
    def quote_if_needed(val):
        if pd.isna(val) or val == '':
            return "''"
        if isinstance(val, str) and re.fullmatch(r'[a-zA-Z0-9_]+', val):
            return val
        else:
            return f"'{val}'"
    outgoing = []
    for idx, row in df.iterrows():
        ref_str = row['Referenced By']
        context = row[context_col] if context_col in df.columns else ''
        if pd.isna(ref_str) or ref_str == '':
            outgoing.append([])
            continue
        items = re.findall(r"'([^']*)'\.'([^']*)'|([a-zA-Z0-9_]+)|'([^']*)'", ref_str)
        refs = []
        for match in items:
            if match[0] and match[1]:
                refs.append(f"{match[0]}.{match[1]}")
            elif match[2]:
                refs.append(match[2])
            elif match[3]:
                refs.append(match[3])
        # нормализация с контекстом
        norm = []
        for r in refs:
            if '.' in r:
                norm.append(r)
            else:
                norm.append(f"{quote_if_needed(context)}.{r}")
        outgoing.append(norm)
    df['Outgoing References'] = outgoing
    df['_Out Refs'] = [[x.replace("'", "") for x in inner] for inner in outgoing]
    return df

def add_ingoing_references(dataframes_dict):
    """
    Добавляет столбец 'Ingoing References' для всех DataFrame в словаре.
    
    Параметры:
        dataframes_dict (dict): словарь вида {имя_файла: pd.DataFrame}
    
    Возвращает:
        dict: тот же словарь с модифицированными датафреймами (изменение in-place)
    """
    # Сбор всех пар (источник → цель) из всех датафреймов
    outgoing_pairs = []
    for df in dataframes_dict.values():
        if 'Full Name' in df.columns and 'Outgoing References' in df.columns:
            for _, row in df.iterrows():
                src = row['Full Name']
                # Обрабатываем 'Outgoing References'
                for tgt in row['Outgoing References']:
                    outgoing_pairs.append((src, tgt))
                # Обрабатываем '_Out Refs' (очищенная версия, если есть)
                for tgt in row.get('_Out Refs', []):
                    outgoing_pairs.append((src, tgt))
    
    # Построение обратного индекса: цель → множество источников
    ingoing_index = defaultdict(set)
    for src, tgt in outgoing_pairs:
        ingoing_index[tgt].add(src)
    
    # Добавление столбца 'Ingoing References' в каждый датафрейм
    for df in dataframes_dict.values():
        if 'Full Name' not in df.columns:
            continue
        ingoing_list = []
        for _, row in df.iterrows():
            cur = row['Full Name']
            sources = ingoing_index.get(cur, set()).copy()
            sources.discard(cur)  # удаляем самоссылки
            ingoing_list.append(sorted(sources))
        df['Ingoing References'] = ingoing_list
    
    return dataframes_dict

print("✅ Все функции очистки загружены")

# === Применение очистки к загруженным данным ===
# Ожидаемые имена файлов (можно адаптировать под ваши)
# Для каждого нужного файла вызываем rename_first_column с соответствующим key_name
dataframes['Multicubes.csv'] = clean_df_multicubes(rename_first_column(dataframes['Multicubes.csv'], 'Multicubes'))
dataframes['Cubes.csv'] = clean_df_cubes(rename_first_column(dataframes['Cubes.csv'], 'Cubes'))
dataframes['Lists.csv'] = clean_df_lists(rename_first_column(dataframes['Lists.csv'], 'Lists'))
dataframes['ListsProperties.csv'] = clean_df_listsproperties(rename_first_column(dataframes['ListsProperties.csv'], 'ListsProperties'))
dataframes['CubesSubsets.csv'] = clean_df_сubesubsets(rename_first_column(dataframes['CubesSubsets.csv'], 'CubesSubsets'))
dataframes['ListsSubsets.csv'] = rename_first_column(dataframes['ListsSubsets.csv'], 'ListsSubsets')

# Обработка Referenced By
dataframes['Cubes.csv'] = clean_referenced_by(dataframes['Cubes.csv'], context_col='Multicube')
dataframes['ListsProperties.csv'] = clean_referenced_by(dataframes['ListsProperties.csv'], context_col='List')

# Добавление входящих ссылок
dataframes = add_ingoing_references(dataframes)

print(f"✅ Очистка завершена!")

## 📄 Часть 2: Генерация отчётов (с возможностью скачивания)

In [ ]:
# === Отчёт 1: report.txt (Multicube – Cubes) ===
def generate_ingoing_report_text(df_multicubes, df_cubes):
    lines = ["📊 ОТЧЁТ: Входящие связи мультикубов и кубов", "="*60, ""]
    for _, row in df_multicubes.iterrows():
        mc = row['Multicubes']
        refs = str(row.get('Ingoing References', []))
        matches = df_cubes[df_cubes['Multicube'].str.contains(mc, na=False, regex=False)]
        if len(matches) == 0:
            continue
        lines.append(f"Мультикуб: {mc} Входящие связи: {refs}")
        for _, mrow in matches.iterrows():
            lines.append(f"    Куб: {mrow['Cubes']}, Входящие свя: {mrow.get('Ingoing References', [])}")
        lines.append("")
    return "\n".join(lines)

report1_content = generate_ingoing_report_text(dataframes['Multicubes.csv'], dataframes['Cubes.csv'])
download_text(report1_content, "report.txt")

In [ ]:
# === Отчёт 2: big_cubes.txt (кубы с Cell Count >= 1_000_000 [или выбранного значения]) ===
def big_cubes_report(df_cubes, threshold=1_000_000, ascending=False):
    df = df_cubes.copy()
    df['Cell Count'] = pd.to_numeric(df['Cell Count'], errors='coerce')
    df = df.dropna(subset=['Cell Count'])
    filtered = df[df['Cell Count'] >= threshold].sort_values('Cell Count', ascending=ascending)
    lines = ["📊 ОТЧЁТ: Большие кубы модели", "="*60, ""]
    for _, row in filtered.iterrows():
        cnt = f"{int(row['Cell Count']):,}"
        lines.append(f"{row['Full Name']} ({cnt})")
        lines.append("")
        lines.append(str(row.get('Ingoing References', [])))
        lines.append("")
    return "\n".join(lines)

report2_content = big_cubes_report(dataframes['Cubes.csv'], cube_threshold_value)
download_text(report2_content, "big_cubes.txt")

In [ ]:
# === Отчёт 3: data_sources.txt (источники данных) ===
def data_sources_report(df_cubes, df_listsproperties):
    lines = ["📊 ОТЧЁТ: Источники данных (сущности без входящих ссылок)", "="*60, ""]
    for df in [df_cubes, df_listsproperties]:
        if 'Full Name' not in df.columns or 'Outgoing References' not in df.columns:
            continue
        for _, row in df.iterrows():
            out = row.get('Outgoing References', [])
            if not isinstance(out, list) or len(out) == 0:
                continue
            ing = row.get('Ingoing References', [])
            # Проверяем, что ing пустой (None, пустой список, пустой массив, NaN)
            is_empty = False
            if ing is None:
                is_empty = True
            elif isinstance(ing, (list, np.ndarray)):
                if len(ing) == 0:
                    is_empty = True
            elif pd.isna(ing):
                is_empty = True
            
            if is_empty:
                cnt = f"{int(row.get('Cell Count', 0)):,}"
                lines.append(f"{row['Full Name']} ({cnt})")
                lines.append("")
                lines.append(str(out))
                lines.append("")
                if 'Referenced By' in df and pd.notna(row.get('Referenced By')):
                    lines.append(str(row['Referenced By']).strip())
                lines.append("")
    return "\n".join(lines)

report3_content = data_sources_report(dataframes['Cubes.csv'], dataframes['ListsProperties.csv'])
download_text(report3_content, "data_sources.txt")

In [ ]:
# === Отчёт 4: multicube_report.txt (обогащённый отчёт с фильтрацией по Cell Count > 700_000) ===
def advanced_multicube_report(cubes_df, multicubes_df, lists_df, subsets_df, threshold=700_000):
    """
    Генерирует отчёт по мультикубам, отбирая кубы с Cell Count > threshold.
    Возвращает строку с отчётом.
    """
    # Подготовка словарей списков и сабсетов
    lists_map = {}
    if 'Lists' in lists_df.columns and 'Cell Count' in lists_df.columns:
        lists_map = lists_df.drop_duplicates('Lists').set_index('Lists')['Cell Count'].to_dict()
    
    subsets_map = {}
    if subsets_df is not None and 'ListsSubsets' in subsets_df.columns and 'Full Name' in subsets_df.columns:
        for _, r in subsets_df.iterrows():
            subsets_map[r['ListsSubsets']] = (r['Full Name'], r.get('Cell Count', 0))
    
    # Копируем и чистим кубы
    cubes_df = cubes_df.copy()
    cubes_df['Cell Count'] = pd.to_numeric(cubes_df['Cell Count'], errors='coerce')
    
    # Фильтруем кубы по порогу
    big_cubes = cubes_df[cubes_df['Cell Count'] > threshold]
    
    # Агрегация: сумма, количество, первое значение
    grouped = big_cubes.groupby('Multicube').agg({'Cell Count': ['sum', 'count', 'first']}).round()
    # Явное приведение к int64 для избежания переполнения (особенно в WebAssembly)
    grouped[('Cell Count', 'sum')] = grouped[('Cell Count', 'sum')].astype('int64')
    grouped[('Cell Count', 'first')] = grouped[('Cell Count', 'first')].astype('int64')
    grouped.columns = ['Sum', 'Count', 'First']
    grouped = grouped.reset_index()
    # Сортировка по убыванию суммы
    grouped = grouped.sort_values('Sum', ascending=False)
    
    # Фильтруем мультикубы по тем, что есть в grouped
    mc_names = set(grouped['Multicube'])
    multicubes_df = multicubes_df[multicubes_df['Multicubes'].isin(mc_names)]
    
    # Парсинг полей с разделителями-запятыми
    def parse_comma(text):
        if not isinstance(text, str):
            return []
        return [x.strip() for x in text.split(',') if x.strip()]
    
    lines = ["📊 ОТЧЁТ: Размерность и источники кубов в мультикубах", "="*60, ""]
    for _, row in grouped.iterrows():
        mc = row['Multicube']
        mc_row = multicubes_df[multicubes_df['Multicubes'] == mc]
        if mc_row.empty:
            continue
        
        # Собираем уникальные элементы из полей мультикуба
        items = set()
        for field in ['User Lists', 'Time Scale', 'Versions', 'Cube Subset']:
            if field in mc_row.columns:
                items.update(parse_comma(mc_row.iloc[0][field]))
        
        # Обогащаем элементы: подставляем Cell Count из списков или Full Name из сабсетов
        enriched = []
        for it in items:
            key = it.strip("'")
            if key in lists_map:
                enriched.append(f"{it} ({lists_map[key]:,})")
            elif key in subsets_map:
                full, cnt = subsets_map[key]
                enriched.append(f"{full} ({cnt:,})")
            else:
                enriched.append(it)
        
        # Формируем строки отчёта
        lines.append(f"{mc} ({row['Sum']:,})")
        lines.append(f"Кубов: {row['Count']}, Ячеек в кубе: {row['First']:,}")
        for e in enriched:
            lines.append(f"    {e}")
        lines.append("=" * 80 + "\n")
    
    return "\n".join(lines)

report4_content = advanced_multicube_report(dataframes['Cubes.csv'], dataframes['Multicubes.csv'], dataframes['Lists.csv'], dataframes['ListsSubsets.csv'], cube_threshold_value)
download_text(report4_content, "multicube_report.txt")

In [ ]:
# === Отчёт 5: формульные дубликаты  ===
def formula_duplication_report(df_list):
    all_rows = []
    for df in df_list:
        if 'Formula' in df.columns and 'Full Name' in df.columns:
            tmp = df[['Formula', 'Full Name']].copy()
            tmp['Formula'] = tmp['Formula'].astype(str).str.strip()
            tmp = tmp[tmp['Formula'] != '']
            all_rows.append(tmp)
    if not all_rows:
        return "Нет данных для анализа формул."
    comb = pd.concat(all_rows, ignore_index=True)
    groups = comb.groupby('Formula')['Full Name'].apply(list).to_dict()
    items = [(formula, sorted(names), len(names)) for formula, names in groups.items()]
    items.sort(key=lambda x: (-x[2], x[0]))
    lines = ["📊 ОТЧЁТ: Уникальные формулы во всех кубах", "="*70, ""]
    for formula, names, cnt in items:
        lines.append(f"Formula: {formula}  (используется в {cnt} кубах)")
        for name in names:
            lines.append(f"    Full Name: {name}")
        lines.append("")
    lines.append(f"📊 Статистика: уникальных формул = {len(items)}, всего кубов с формулами = {len(comb)}")
    return "\n".join(lines)

report5_content = formula_duplication_report([dataframes['Cubes.csv'], dataframes['ListsProperties.csv']])
download_text(report5_content, "formula_duplicates.txt")

## 📈 Часть 2: Анализ формул и формульных зависимостей

In [ ]:
# === Подготовка данных для графа ===

# Словарь для ListsProperties
lists_properties_map = {}
if 'ListsProperties.csv' in dataframes:
    df_lp = dataframes['ListsProperties.csv']
    if 'ListsProperties' in df_lp.columns:
        for _, row in df_lp.iterrows():
            key = str(row['ListsProperties']).strip()
            if 'Full Name' in df_lp.columns and pd.notna(row['Full Name']) and row['Full Name'] != '':
                value = row['Full Name']
            else:
                value = key
                # Если значение содержит не только буквы/цифры/_, и не обёрнуто в кавычки, оборачиваем
                if not re.match(r'^[a-zA-Z0-9_]+$', key) and not (key.startswith("'") and key.endswith("'")):
                    value = f"'{key}'"
            lists_properties_map[key] = value

# Словарь для CubesSubsets
cubes_subsets_map = {}
if 'CubesSubsets.csv' in dataframes:
    df_cs = dataframes['CubesSubsets.csv']
    if 'CubesSubsets' in df_cs.columns:
        for _, row in df_cs.iterrows():
            key = str(row['CubesSubsets']).strip()
            if 'Full Name' in df_cs.columns and pd.notna(row['Full Name']) and row['Full Name'] != '':
                value = row['Full Name']
            else:
                value = key
                if not re.match(r'^[a-zA-Z0-9_]+$', key) and not (key.startswith("'") and key.endswith("'")):
                    value = f"'{key}'"
            cubes_subsets_map[key] = value

# Словарь для ListsSubsets
lists_subsets_map = {}
if 'ListsSubsets.csv' in dataframes:
    df_ls = dataframes['ListsSubsets.csv']
    if 'ListsSubsets' in df_ls.columns:
        for _, row in df_ls.iterrows():
            key = str(row['ListsSubsets']).strip()
            if 'Full Name' in df_ls.columns and pd.notna(row['Full Name']) and row['Full Name'] != '':
                value = row['Full Name']
            else:
                value = key
                if not re.match(r'^[a-zA-Z0-9_]+$', key) and not (key.startswith("'") and key.endswith("'")):
                    value = f"'{key}'"
            lists_subsets_map[key] = value

# Приводим df_cubes к нужному виду: столбцы Cube, Multicube, Formula
df_graph = dataframes['Cubes.csv'].rename(columns={'Cubes': 'Cube'})
# Создаём множества списков (справочники) из df_lists
lists_set = set(dataframes['Lists.csv']['Lists'].dropna().unique()) if 'Lists' in dataframes['Lists.csv'].columns else set()
# Построение словарей
cube_to_multicube = {}
multicube_to_cubes = defaultdict(list)
for _, row in df_graph.iterrows():
    cube = row['Cube']
    mc = row['Multicube']
    if cube and pd.notna(cube) and cube != '':
        if cube not in cube_to_multicube:
            cube_to_multicube[cube] = mc
        multicube_to_cubes[mc].append(cube)
full_cube_names = set(df_graph['Full Name'].unique())

# === Парсер для парсинга ссылок из формул
def make_optimized_parser(lists_set, cube_to_multicube, multicube_to_cubes):
    """
    Создаёт оптимизированную функцию парсинга формул.
    Все регулярные выражения компилируются один раз здесь.
    
    Параметры:
        lists_set: set — имена списков (справочников)
        cube_to_multicube: dict — отображение имени куба в мультикуб
        multicube_to_cubes: dict — отображение мультикуба в список кубов
    """
    # ========= Шаг 1: FINDITEM =========
    _finditem_re = re.compile(
        r"FINDITEM\s*\(\s*'([^']*)'\s*,\s*'([^']*)'\s*\)",
        re.IGNORECASE
    )
    
    # ========= Шаг 2: двойные сущности 'multicube'.'cube' =========
    double_templates = []
    double_mapping = {}
    for mc, cubes in multicube_to_cubes.items():
        for cube in cubes:
            template = f"'{mc}'.'{cube}'"
            double_templates.append(re.escape(template))
            double_mapping[template] = f"{mc}.{cube}"
    _double_re = re.compile("|".join(double_templates)) if double_templates else None
    
    # ========= Общий regex для любых двойных кавычек =========
    _any_double_re = re.compile(r"'([^']*)'\.'([^']*)'")
    
    # ========= Шаг 6: ссылки на мультикубы (одинарные) =========
    if multicube_to_cubes:
        mc_alt = "|".join(re.escape(name) for name in multicube_to_cubes.keys())
        _multicube_re = re.compile(r"'({})'".format(mc_alt))
    else:
        _multicube_re = None
    
    # ========= Шаг 7: ссылки на списки =========
    if lists_set:
        lists_alt = "|".join(re.escape(lst) for lst in lists_set)
        _lists_re = re.compile(r"'({})'".format(lists_alt))
    else:
        _lists_re = None
    
    # ========= Шаг 8: одинарные имена (артефакты) =========
    _single_re = re.compile(r"'([^']*)'")
    
    # ========= Внутренняя функция парсинга =========
    def parse_formula(formula, current_multicube, current_cube_name,
                      lists_properties_map=None,
                      cubes_subsets_map=None,
                      lists_subsets_map=None):
        if not isinstance(formula, str) or pd.isna(formula):
            return []
        
        # Удаляем комментарии TC(...)
        formula = re.sub(r'TC\([^)]*\)', '__TC_PLACEHOLDER__', formula)
        # Удаляем обрамление
        double_quoted = r'^\s*"\'"\s*&\s*|\s*&\s*"\'"\s*$'
        working = re.sub(double_quoted, '', formula)
        references = []
        
        # ----- Шаг 1: FINDITEM -----
        def finditem_cb(m):
            lst = m.group(1).strip()
            item = m.group(2).strip()
            if lst and item:
                references.append(f"{lst}.{item}")
            return "FINDITEM_PLACEHOLDER"
        working = _finditem_re.sub(finditem_cb, working)
        
        # ----- Шаг 2: двойные сущности 'multicube'.'cube' -----
        if _double_re:
            def double_cb(m):
                ref = double_mapping.get(m.group(0))
                if ref:
                    references.append(ref)
                return "DOUBLE_PLACEHOLDER"
            working = _double_re.sub(double_cb, working)
        
        # ----- Шаг 3: ссылки на элементы списков 'list'.'item' -----
        known_list_items = set()
        def list_item_cb(m):
            lst = m.group(1).strip()
            item = m.group(2).strip()
            if lst in lists_set and item:
                full = f"{lst}.{item}"
                references.append(full)
                known_list_items.add(full)
                return "LIST_ITEM_PLACEHOLDER"
            return m.group(0)
        working = _any_double_re.sub(list_item_cb, working)
        
        # ----- Шаг 4: неизвестные двойные сущности -----
        def unknown_double_cb(m):
            text1 = m.group(1).strip()
            text2 = m.group(2).strip()
            full = f"{text1}.{text2}"
            if full not in known_list_items and text1 and text2:
                references.append(full)
                return "UNKNOWN_DOUBLE_PLACEHOLDER"
            return m.group(0)
        working = _any_double_re.sub(unknown_double_cb, working)
        
        # ----- Шаг 5: внутренние кубы текущего мультикуба -----
        if current_multicube in multicube_to_cubes:
            cubes_in_current = multicube_to_cubes[current_multicube]
            if cubes_in_current:
                cubes_alt = "|".join(re.escape(c) for c in cubes_in_current)
                _internal_re = re.compile(r"'({})'".format(cubes_alt))
                def internal_cb(m):
                    cube_name = m.group(1)
                    references.append(f"{current_multicube}.{cube_name}")
                    return "INTERNAL_CUBE_PLACEHOLDER"
                working = _internal_re.sub(internal_cb, working)
        
        # ----- Шаг 6: ссылки на мультикубы -----
        if _multicube_re:
            def multicube_cb(m):
                references.append(m.group(1))
                return "MULTICUBE_PLACEHOLDER"
            working = _multicube_re.sub(multicube_cb, working)
        
        # ----- Шаг 7: ссылки на списки -----
        if _lists_re:
            def list_cb(m):
                references.append(m.group(1))
                return "LIST_PLACEHOLDER"
            working = _lists_re.sub(list_cb, working)
        
        # ----- Шаг 8: артефакты (одинарные имена) -----
        def artifact_cb(m):
            artifact = m.group(1).strip()
            if not artifact:
                return m.group(0)
            # Пропускаем плейсхолдеры
            if artifact.startswith(("FINDITEM_PLACEHOLDER", "DOUBLE_PLACEHOLDER",
                                     "LIST_ITEM_PLACEHOLDER", "UNKNOWN_DOUBLE_PLACEHOLDER",
                                     "INTERNAL_CUBE_PLACEHOLDER", "MULTICUBE_PLACEHOLDER",
                                     "LIST_PLACEHOLDER")):
                return m.group(0)
            # Проверка дополнительных справочников
            if lists_properties_map and artifact in lists_properties_map:
                references.append(lists_properties_map[artifact])
            elif cubes_subsets_map and artifact in cubes_subsets_map:
                references.append(cubes_subsets_map[artifact])
            elif lists_subsets_map and artifact in lists_subsets_map:
                references.append(lists_subsets_map[artifact])
            else:
                references.append(f"External.{artifact}")
            return "EXTERNAL_ARTIFACT_PLACEHOLDER"
        working = _single_re.sub(artifact_cb, working)
        
        # ========= Финальная очистка =========
        # Удаление дубликатов с сохранением порядка
        unique = []
        seen = set()
        for ref in references:
            if ref not in seen:
                seen.add(ref)
                unique.append(ref)
        
        # Удаление текущего куба
        if current_cube_name in unique:
            unique.remove(current_cube_name)
        
        # Удаление мультикубов, если присутствуют их кубы
        filtered = []
        for ref in unique:
            if ref in multicube_to_cubes:
                if any(item.startswith(ref + '.') for item in unique):
                    continue
            filtered.append(ref)
        
        # Удаление списков, если присутствуют их элементы
        final = []
        for ref in filtered:
            if ref in lists_set:
                if any(item.startswith(ref + '.') for item in filtered):
                    continue
            final.append(ref)
        
        return final
    
    return parse_formula

print("✅ Функция разбора формул загружена")

In [ ]:
# После загрузки данных и построения словарей
optimized_parser = make_optimized_parser(
    lists_set=lists_set,
    cube_to_multicube=cube_to_multicube,
    multicube_to_cubes=multicube_to_cubes
)

# Применяем к каждому кубу
df_graph['ParsedReferences'] = df_graph.apply(
    lambda row: optimized_parser(
        formula=row.get('Formula', ''),
        current_multicube=row['Multicube'],
        current_cube_name=row['Cube'],
        lists_properties_map=lists_properties_map,
        cubes_subsets_map=cubes_subsets_map,
        lists_subsets_map=lists_subsets_map
    ), axis=1
)

print(f"✅ Зависимости извлечены. Всего кубов с формулами: {len(df_graph[df_graph['ParsedReferences'].map(len) > 0])}")

In [ ]:
def report_cube_references(df_graph):
    # Фильтруем и сортируем
    filtered = df_graph[df_graph['ParsedReferences'].map(len) > 0]
    filtered = filtered.sort_values('ParsedReferences', key=lambda x: x.map(len), ascending=False)
    
    lines = ["📊 ОТЧЁТ: Кубы и используемые ими ссылки", "="*80, ""]
    for _, row in filtered.iterrows():
        cube = row['Full Name']
        refs = row['ParsedReferences']
        formula = row.get('Formula', '')  # если нет столбца, будет пустая строка
        lines.append(f"Куб: {cube}")
        lines.append("")
        lines.append(f"Формула: {formula}")
        lines.append("")
        lines.append(f"Ссылки: {refs}")
        lines.append("")
        lines.append("")
    return "\n".join(lines)

# Генерация и скачивание
report_content = report_cube_references(df_graph)
download_text(report_content, "cube_references_with_formulas.txt")

In [ ]:
from collections import OrderedDict
import math

def build_full_name_mapping(dataframes):
    """
    Собирает словарь {Full Name: Cell Count} из всех датафреймов,
    у которых есть колонки 'Full Name' и 'Cell Count'.
    Если один Full Name встречается в нескольких местах, берётся первое значение.
    """
    mapping = {}
    for name, df in dataframes.items():
        if 'Full Name' in df.columns and 'Cell Count' in df.columns:
            for _, row in df.iterrows():
                fn = row['Full Name']
                if pd.notna(fn) and fn != '' and fn not in mapping:
                    try:
                        cnt = float(row['Cell Count'])
                        if math.isnan(cnt):
                            cnt = 0
                        mapping[fn] = int(cnt)
                    except (ValueError, TypeError):
                        mapping[fn] = 0
    return mapping

def product_report(df_graph, full_name_mapping):
    lines = ["📊 ОТЧЁТ: Произведение Cell Count зависимостей по кубам", "=" * 80, ""]
    
    cubes_with_refs = df_graph[df_graph['ParsedReferences'].map(len) > 0]
    entries = []
    
    for _, row in cubes_with_refs.iterrows():
        cube_name = row['Full Name']
        refs = row['ParsedReferences']
        found = {}
        for ref in refs:
            if ref in full_name_mapping and ref not in found:
                found[ref] = full_name_mapping[ref]
        if not found:
            continue
        product = 1
        for cnt in found.values():
            product *= cnt
        entries.append((product, cube_name, found))
    
    # Сортировка по убыванию произведения
    entries.sort(key=lambda x: x[0], reverse=True)
    
    for product, cube_name, found in entries:
        product_str = f"{product:,}"
        lines.append(f"{cube_name} ({product_str})")
        for ref, cnt in found.items():
            cnt_str = f"{cnt:,}"
            lines.append(f"    {ref} ({cnt_str})")
        lines.append("")
    
    if not entries:
        lines.append("Нет кубов с найденными зависимостями.")
    
    return "\n".join(lines)

# Использование
full_name_mapping = build_full_name_mapping(dataframes)
report_product = product_report(df_graph, full_name_mapping)
download_text(report_product, "product_report.txt")

In [ ]:
# === Построение графа ===
G = nx.DiGraph()
for node in full_cube_names:
    G.add_node(node)
for _, row in df_graph.iterrows():
    src = row['Full Name']
    for target in row['ParsedReferences']:
        G.add_edge(target, src)   # источник → потребитель
print(f"✅ Граф построен: узлов = {len(G.nodes)}, рёбер = {len(G.edges)}")

In [ ]:
# === Проверка на циклы ===
double_nodes = {n for n in G.nodes() if isinstance(n, str) and n.count('.') == 1}
G_double = G.subgraph(double_nodes).copy()
if nx.is_directed_acyclic_graph(G_double):
    print("✅ Циклы между кубами не обнаружены.")
else:
    cycles = list(nx.simple_cycles(G_double))
    print(f"⚠️ Обнаружены циклы ({len(cycles)}):")
    for cyc in cycles[:5]:
        print(" → ".join(cyc + [cyc[0]]))

## ✳ Построение связей куба графически

Введите в поле "Ввод" полное имя куба (`'Имя мультикуба'.'Имя куба'`) и нажмите "Применить"

In [ ]:
# Создаём текстовое поле
text_input = widgets.Text(
    value='',           # пустая строка по умолчанию
    placeholder='Введите значение...',
    description='Ввод:',
    disabled=False
)

button = widgets.Button(description='Применить')
output = widgets.Output()

def on_button_click(b):
    with output:
        output.clear_output()
        val = text_input.value
        print(f"Получено значение: '{val}'")
        # === Визуализация окрестности выбранного куба ===
        target_cube = val
        if target_cube in G.nodes:
            # Получаем окрестность глубиной 1
            neighbors = set(G.predecessors(target_cube)) | set(G.successors(target_cube))
            neighbors.add(target_cube)
            subG = G.subgraph(neighbors).copy()
            plt.figure(figsize=(12, 8))
            pos = nx.spring_layout(subG, seed=42)
            node_colors = ['gold' if n == target_cube else 'lightblue' for n in subG.nodes()]
            nx.draw(subG, pos, with_labels=True, node_color=node_colors, node_size=2000, font_size=8, arrows=True)
            plt.title(f"Окрестность куба {target_cube}")
            plt.show()
        else:
            print(f"Куб '{target_cube}' не найден в графе. Доступные кубы (первые 10): {list(G.nodes)[:10]}")
        

button.on_click(on_button_click)
display(text_input, button, output)
